<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.4.1. Finding the Minimal Neural Network for XOR
  </strong>
</div>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from rich.table import Table
from rich.console import Console

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

Rich console instance to display results

In [ ]:
console = Console()

## Data Loading

In [ ]:
## XOR Data

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

## Preprocessing of data

Method to convert to tensors

In [ ]:
def to_tensors(X, y):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)

    return X_tensor, y_tensor

### Configurations

In [ ]:
architectures = {
    "1-Layer (2)": [2],
    "1-Layer (8)": [8],
    "1-Layer (32)": [32],
    "2-Layers (4-8)": [4, 8],
    "2-Layers (8-8)": [8, 8],
    "2-Layers (32-16)": [32, 16],
    "3-Layers (2-4-8)": [2, 4, 8],
    "3-Layers (32-16-8)": [32, 16, 8],
    "3-Layers (64-32-16)": [64, 32, 16],
    "4-Layers (16-16-8-4)": [16, 16, 8, 4],
    "4-Layers (32-32-16-8)": [32, 32, 16, 8],
}

### Model Architectures

In [ ]:
class XORNeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(XORNeuralNetwork, self).__init__()

        layers = []
        last_size = input_size

        for h in hidden_sizes:
            layers.append(nn.Linear(last_size, h))
            layers.append(nn.ReLU())
            last_size = h

        layers.append(nn.Linear(last_size, output_size))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

### Training Methods

In [ ]:
def train_nn_model(model, X_tensor, y_tensor, epochs, lr):
    loss_history = []
    accuracy_history = []

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        y_logits = model(X_tensor)
        loss = criterion(y_logits, y_tensor)

        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

        with torch.no_grad():
            y_pred = torch.sigmoid(y_logits).round()
            acc = accuracy_score(y_tensor.cpu().numpy(), y_pred.cpu().numpy())
            accuracy_history.append(acc)

    return model, loss_history, accuracy_history

## Evaluation Methods

Method to evaluate the neural network

In [ ]:
def evaluate_nn_model(model, X_tensor):
    model.eval()
    with torch.no_grad():
        y_logits = model(X_tensor)
        y_pred = (torch.sigmoid(y_logits) >= 0.5).int().numpy()

    return y_pred

Method to calculate metrics

In [ ]:
def calculate_metrics(y, y_pred):
    acc = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    return {
        "accuracy_score": acc,
        "precision_score": precision,
        "recall_score": recall,
        "f1_score": f1,
    }

## Visualization

Method to create the comparison table

In [ ]:
def create_comparison_table(title, results):

    comparison_table = Table(title=title)
    comparison_table.add_column("Model", justify="center", style="green", width=15)
    comparison_table.add_column("Accuracy", justify="right", style="green")
    comparison_table.add_column("Precision", justify="right", style="green")
    comparison_table.add_column("Recall", justify="right", style="green")
    comparison_table.add_column("F1", justify="right", style="green")

    for model, info in results.items():
        metrics = info["metrics"]
        comparison_table.add_row(
            model,
            f'{metrics["accuracy_score"]:.4f}',
            f'{metrics["precision_score"]:.4f}',
            f'{metrics["recall_score"]:.4f}',
            f'{metrics["f1_score"]:.4f}',
        )

    console.print(comparison_table)

Method to graph loss curves.

In [ ]:
def plot_loss_curves(training_curves):

    n_cols = 4
    n_rows = 3

    plt.figure(figsize=(20, 15))

    for i, (label, curves) in enumerate(training_curves.items(), 1):
        plt.subplot(n_rows, n_cols, i)
        plt.plot(curves["loss"], label="Loss", color="blue")
        plt.plot(curves["accuracy"], label="Accuracy", color="orange")
        plt.title(label, fontsize=10)
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.grid(True)

        plt.legend(loc="best", fontsize=8)

    plt.tight_layout()
    plt.show()

## 1. Create a neural network XOR with different network architectures.

During testing, I saw that activation functions such as ReLU and Sigmoid, loss functions such as BCELoss or BCEWithLogitsLoss, and optimizers such as Adam or AdamW are often used to solve XOR.

After experimenting, I chose:

- ReLU as the activation function because it makes the network learn faster and can handle the non-linearity of the problem.
- BCEWithLogitsLoss as the loss function because it already has the sigmoid function built in and is more stable for this type of binary classification.
- Adam as the optimizer because it automatically adjusts the learning steps and usually works well in most cases.
- A learning rate of 0.001, as this was a value that allowed the network to learn steadily without the loss rising too much.
- 2000 epochs, which were sufficient for the network to learn correctly without the need for much longer training.

Once I had chosen these parameters, I tried out several hidden layers to see what changes would occur:

In [ ]:
learning_rate = 0.001
epochs = 2000

model_metrics = {}
training_curves = {}

X_tensor, y_tensor = to_tensors(X, y)

for arch_name, hidden_sizes in architectures.items():
    model = XORNeuralNetwork(2, hidden_sizes, 1)
    model, loss_history, accuracy_history = train_nn_model(
        model, X_tensor, y_tensor, epochs, learning_rate
    )

    y_pred = evaluate_nn_model(model, X_tensor)

    metrics = calculate_metrics(y, y_pred)

    model_metrics[arch_name] = {
        "metrics": metrics,
        "loss": loss_history,
        "accuracy": accuracy_history,
    }

## 2. Training curves for successful architectures

In [ ]:
for arch_name, info in model_metrics.items():

    metrics = info["metrics"]
    loss_history = info["loss"]
    accuracy_history = info["accuracy"]

    if metrics["accuracy_score"] == 1.00:
        training_curves[f"{arch_name}"] = {
            "loss": loss_history,
            "accuracy": accuracy_history,
        }

plot_loss_curves(training_curves)

## 3. Table comparing architectures

In [ ]:
create_comparison_table(
    "Comparison of metrics between different architectures", model_metrics
)

When comparing the models, it can be seen that several achieve perfect performance, but the `1-Layer (8) model` turns out to be the best option, as it achieves the same level of performance while being the smallest and simplest. This makes it more efficient in its use of resources, consistent in its results, and avoids adding unnecessary complexity that does not provide additional improvements.

## 3. Decision boundary of the smallest and most consistent neural network architecture found

In [ ]:
learning_rate = 0.001
epochs = 2000

X_tensor, y_tensor = to_tensors(X, y)

best_architecture = [8]

model = XORNeuralNetwork(input_size=2, hidden_sizes=best_architecture, output_size=1)
model, _, _ = train_nn_model(model, X_tensor, y_tensor, epochs, learning_rate)

In [ ]:
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01), np.arange(y_min, y_max, 0.01))
grid = np.c_[xx.ravel(), yy.ravel()]

grid_tensor = torch.tensor(grid, dtype=torch.float32)

with torch.no_grad():
    preds = torch.sigmoid(model(grid_tensor))
preds = (preds >= 0.5).numpy().reshape(xx.shape)

plt.contourf(xx, yy, preds, alpha=0.3, cmap=plt.cm.coolwarm)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors="k", s=20)
plt.title("XOR Neural Network Decision Boundary")
plt.xlabel("Input 1")
plt.ylabel("Input 2")
plt.show()

## 4. Is there a balance between your choice of architecture and the number of parameters?

Yes, there is a balance between the architecture we choose and the number of parameters. I can explain it with the following analogy, since it's the one I liked the most, the problem is a cooking recipe, the ingredients are like the parameters and the architecture is where they go, so if we use too few ingredients, or if they are not the right ones, the end result will not be what we expected. But if we use too many ingredients, it can also ruin the recipe. It may end up too salty, too sweet, or simply heavier than necessary.

On the other hand, a large pot (a very complex architecture) does not always guarantee a better result. Sometimes, with a simpler structure, we can get what we want more quickly and efficiently.